### NEO Risk Analysis

**Course:** CPE-551

This notebook analyzes near-Earth asteroid data from NASA's NeoWs API and applies a custom risk scoring system that improves on NASA's binary hazardous/non-hazardous classification by weighting asteroid diameter, velocity, and miss distance into a graduated 0-10 score.

### 1. Imports, Setup and Data Load

In [1]:
import math
import json
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

from asteroid import Asteroid
from asteroid_tracker import AsteroidTracker
from functions import calculate_risk_score, normalize_value

plt.rcParams["figure.figsize"] = (10, 6)

try:
    tracker = AsteroidTracker("data/data.json")
    print(f"Loaded {len(tracker)} asteroids from data/data.json")
except FileNotFoundError as e:
    print(f"Data file not found: {e}")
    raise
except json.JSONDecodeError as e:
    print(f"Could not parse JSON: {e}")
    raise

Loaded 92 asteroids from data/data.json


### 2. Basic Stats

Total asteroid count, date range, NASA hazardous count, and risk-category breakdown. Along with showcasing the date span via `datetime`. Compute the number of days covered by the dataset using the standard library `datetime` module

In [2]:
stats = tracker.get_summary_stats()

print("Dataset Summary")
print("-" * 40)
print(f"Total asteroids:    {stats['total_count']}")
print(f"Date range:         {stats['date_range'][0]} to {stats['date_range'][1]}")
print(f"NASA hazardous:     {stats['hazardous_count']}")
print(f"Mean risk score:    {stats['mean_risk']:.2f} / 10")
print(f"Max risk score:     {stats['max_risk']:.2f} / 10")
print(f"Min risk score:     {stats['min_risk']:.2f} / 10")
print("\nRisk category counts:")
for category, count in stats["category_counts"].items():
    print(f"  {category:<10} {count}")
start_str, end_str = stats["date_range"]
start_dt = datetime.strptime(start_str, "%Y-%m-%d")
end_dt = datetime.strptime(end_str, "%Y-%m-%d")
span_days = (end_dt - start_dt).days + 1
print(f"Dataset spans {span_days} days ({start_str} to {end_str}).")

Dataset Summary
----------------------------------------
Total asteroids:    92
Date range:         2026-04-04 to 2026-04-11
NASA hazardous:     5
Mean risk score:    2.98 / 10
Max risk score:     8.08 / 10
Min risk score:     0.59 / 10

Risk category counts:
  Low        36
  Medium     48
  High       7
  Critical   1


### 3. Top 10 Riskiest Asteroids

`get_top_risks(n)` sorts the collection by computed risk score and returns the top `n`. Each `Asteroid` defines `__str__` for a readable one-line summary and `__gt__` so comparisons work directly on score.

In [4]:
top10 = tracker.get_top_risks(10)

for i, asteroid in enumerate(top10, start=1):
    print(f"{i:>2}. {asteroid}")

 1. Asteroid 302831 (2003 FH) | Diameter: 645.0m | Miss: 27762566.7 km | Risk: 8.08 (Critical)
 2. Asteroid 515049 (2010 FL) | Diameter: 593.7m | Miss: 24074792.9 km | Risk: 7.49 (High)
 3. Asteroid (2006 GC1) | Diameter: 341.6m | Miss: 13278868.8 km | Risk: 7.26 (High)
 4. Asteroid (2018 PG21) | Diameter: 556.6m | Miss: 15335437.1 km | Risk: 6.96 (High)
 5. Asteroid 399325 (1999 GY5) | Diameter: 395.9m | Miss: 10534194.3 km | Risk: 6.78 (High)
 6. Asteroid 363599 (2004 FG11) | Diameter: 265.2m | Miss: 8454614.0 km | Risk: 6.42 (High)
 7. Asteroid (2026 FL5) | Diameter: 163.1m | Miss: 14995390.5 km | Risk: 5.34 (High)
 8. Asteroid (2026 FG5) | Diameter: 42.3m | Miss: 3296996.2 km | Risk: 5.23 (High)
 9. Asteroid (2022 GR2) | Diameter: 84.6m | Miss: 37984978.2 km | Risk: 4.98 (Medium)
10. Asteroid (2020 HK7) | Diameter: 184.3m | Miss: 38044039.6 km | Risk: 4.88 (Medium)


### 4. Build a DataFrame for Plotting

Use a list comprehension to extract per-asteroid fields, then load them into a pandas `DataFrame`. This makes the plots in the next sections easy to produce.

In [5]:
rows = [
    {
        "name": a.name,
        "avg_diameter_m": (a.diameter_min + a.diameter_max) / 2,
        "velocity_kmh": a.velocity,
        "miss_distance_km": a.miss_distance,
        "is_hazardous": a.is_hazardous,
        "close_approach_date": a.close_approach_date,
        "risk_score": a.calculate_risk_score(),
        "risk_category": a.get_risk_category(),
    }
    for a in tracker.asteroids
]

df = pd.DataFrame(rows)
df.head()

,name,avg_diameter_m,velocity_kmh,miss_distance_km,is_hazardous,close_approach_date,risk_score,risk_category
0,330659 (2008 GG2),107.533212,25884.956391,2.205655e+07,False,2026-04-11,3.096559,Medium
1,363599 (2004 FG11),265.181130,90256.629018,8.454614e+06,True,2026-04-11,6.422275,High
2,(2017 GS6),47.156614,62304.840837,7.146886e+07,False,2026-04-11,1.687149,Low
3,(2017 UR5),7.826050,30603.312274,5.696653e+07,False,2026-04-11,1.163657,Low
4,(2019 GK3),5.414304,19938.987623,1.289511e+07,False,2026-04-11,2.673278,Medium
